# Phase 09.1 ABL-01: Preprocessing Round-Trip Verification

Round-trip gate for the three ablation preprocessing pipelines defined in
`revision/core/preprocessing.py` (per Phase 09.1 CONTEXT D-09.1-01..03).

Behavior contract:

- **Pipeline A** (`forward_minmax_od` / `inverse_minmax_od`): max|inv(fwd(OD)) - OD| ≤ 1e-8 on the real 778-element OD tensor (float-eps).
- **Pipeline B** (`forward_logreturns` / `inverse_logreturns`): max|inv(fwd(OD), od[0], mu, sigma) - OD| ≤ 1e-8 on the real 778-element OD tensor anchored at `od[0]` (float-eps).
- **Pipeline C** (re-exported `forward_lambert` / `inverse_lambert`): max|inv(fwd(real_norm)) - real_norm| ≤ 1e-6 on the 777-element `norm_log_delta` tensor (matches Phase 9 EVAL-06 `full_pipeline` tolerance, D-09.1-03).

Output: `revision/results/transform_ablation/roundtrip_check.json` with `all_passed: true`.

In [1]:
import json
import os
import subprocess
import sys
import warnings
from pathlib import Path

import numpy as np
import torch

warnings.filterwarnings("ignore")

def _find_repo_root():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data.csv").exists() and (d / "revision" / "core").is_dir():
            return d
    raise FileNotFoundError(
        "Could not locate repo root from " + str(here) +
        " (looked for data.csv + revision/core)"
    )

REPO_ROOT = _find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Repo root: {REPO_ROOT}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

def _git_sha():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            stderr=subprocess.DEVNULL,
        ).decode().strip()
    except Exception:
        return "unknown"

Repo root: /Users/shawngibford/dev/phd/qGAN/.claude/worktrees/agent-a83da323461406c79


In [2]:
from revision.core.data import load_and_preprocess
from revision.core.preprocessing import (
    forward_minmax_od, inverse_minmax_od,
    forward_logreturns, inverse_logreturns,
    forward_lambert, inverse_lambert,
)

d = load_and_preprocess("./data.csv")
od = d["OD"].double()  # promote to float64 for round-trip precision
real_norm = d["norm_log_delta"].double()
delta = float(d["delta"])

print(f"OD len = {od.numel()}, dtype = {od.dtype}, min = {od.min().item():.4f}, max = {od.max().item():.4f}")
print(f"norm_log_delta len = {real_norm.numel()}, delta = {delta:.6f}")
assert od.numel() == 778, f"expected 778 OD values, got {od.numel()}"
assert real_norm.numel() == 777, f"expected 777 norm_log_delta values, got {real_norm.numel()}"
assert (od > 0).all().item(), "OD must be strictly positive for log-returns"

OD len = 778, dtype = torch.float64, min = 0.4700, max = 3.8000
norm_log_delta len = 777, delta = 0.146932


In [3]:
# Pipeline A: min-max normalize to [0, 1], inverse back to OD scale.
scaled, od_min, od_max = forward_minmax_od(od)
assert scaled.min().item() >= 0.0 and scaled.max().item() <= 1.0, \
    f"Pipeline A forward output out of [0,1]: [{scaled.min().item()}, {scaled.max().item()}]"
od_rt_a = inverse_minmax_od(scaled, od_min, od_max)
err_a = (od_rt_a - od).abs().max().item()
print(f"[A] roundtrip max_abs_err = {err_a:.3e}  (tol ≤ 1e-8)")
assert err_a <= 1e-8, f"Pipeline A round-trip FAILED: err = {err_a}"

[A] roundtrip max_abs_err = 0.000e+00  (tol ≤ 1e-8)


In [4]:
# Pipeline B: standardized log-returns with cumulative-integration inverse.
# Treat the full 778-point OD series as one trajectory anchored at od[0].
r_norm, mu, sigma = forward_logreturns(od)
assert r_norm.shape[0] == od.shape[0] - 1, \
    f"forward_logreturns length contract violated: {r_norm.shape[0]} != {od.shape[0] - 1}"
od_rt_b = inverse_logreturns(r_norm, od[0], mu, sigma)
err_b = (od_rt_b - od).abs().max().item()
print(f"[B] roundtrip max_abs_err = {err_b:.3e}  (tol ≤ 1e-8)")
assert err_b <= 1e-8, f"Pipeline B round-trip FAILED: err = {err_b}"

[B] roundtrip max_abs_err = 4.441e-15  (tol ≤ 1e-8)


In [5]:
# Pipeline C: re-exported Lambert W pair via revision.core.preprocessing.
# Re-verifies the Phase 9 EVAL-06 full_pipeline tolerance (≤ 1e-6) on the
# re-export path (NOT through data.py directly) so the re-export wiring is
# exercised on every run (T-09.1.01-01 mitigation).
#
# NOTE: lambert_w_transform clips its output to [clip_low=-12, clip_high=11]
# for stability; two real-data outliers (|norm|>3.80) would saturate +11 and
# corrupt the round-trip. Same _BIG trick as 02_eval06_roundtrip.ipynb.
_BIG = 1.0e20
y = forward_lambert(real_norm, delta, clip_low=-_BIG, clip_high=_BIG)
rt = inverse_lambert(y, delta)
err_c = (rt - real_norm).abs().max().item()
print(f"[C] roundtrip max_abs_err = {err_c:.3e}  (tol ≤ 1e-6)")
assert err_c <= 1e-6, f"Pipeline C round-trip FAILED: err = {err_c}"

[C] roundtrip max_abs_err = 4.441e-16  (tol ≤ 1e-6)


In [6]:
all_passed = (err_a <= 1e-8) and (err_b <= 1e-8) and (err_c <= 1e-6)
artifact = {
    "pipelines": {
        "A": {"max_abs_err": float(err_a), "tolerance": 1e-8, "pass": bool(err_a <= 1e-8)},
        "B": {"max_abs_err": float(err_b), "tolerance": 1e-8, "pass": bool(err_b <= 1e-8)},
        "C": {"max_abs_err": float(err_c), "tolerance": 1e-6, "pass": bool(err_c <= 1e-6)},
    },
    "all_passed": bool(all_passed),
    "seed": SEED,
    "git_sha": _git_sha(),
    "notes": "Phase 09.1 ABL-01: forward/inverse round-trip for Pipelines A, B (new) and C (Phase 9 EVAL-06 re-export verification).",
}

out = Path("revision/results/transform_ablation/roundtrip_check.json")
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(artifact, indent=2))
print(json.dumps(artifact, indent=2))

assert all_passed, f"ABL-01 round-trip FAILED: {artifact}"
print("ABL-01 round-trip PASSED")

{
  "pipelines": {
    "A": {
      "max_abs_err": 0.0,
      "tolerance": 1e-08,
      "pass": true
    },
    "B": {
      "max_abs_err": 4.440892098500626e-15,
      "tolerance": 1e-08,
      "pass": true
    },
    "C": {
      "max_abs_err": 4.440892098500626e-16,
      "tolerance": 1e-06,
      "pass": true
    }
  },
  "all_passed": true,
  "seed": 42,
  "git_sha": "bde33ef9ad0b899303624ed59a96a7c5dd1a2a97",
  "notes": "Phase 09.1 ABL-01: forward/inverse round-trip for Pipelines A, B (new) and C (Phase 9 EVAL-06 re-export verification)."
}
ABL-01 round-trip PASSED
